[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/01_%E5%8C%BA%E9%96%93%E6%8E%A8%E5%AE%9A.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。そのまま上から順に実行できます。

# 統計2級-01. 区間推定（信頼区間）

**🎯 できるようになること**
- 点推定と区間推定の違いを説明できる
- 母平均(t分布)・母比率の信頼区間を計算できる
- 信頼度やnで区間の幅がどう変わるか説明できる

**前提**：統計3級04（標本・標準誤差）　/　**所要**：約30分　/　**レベル**：統計検定2級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数（分布・検定など）
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
rng = np.random.default_rng(0)   # 乱数生成器（seedで結果を固定）

## 1. 点推定 と 区間推定

- **点推定**：標本平均をそのまま母平均の推定値とする（1つの値）
- **区間推定**：「母平均はこの範囲にありそう」と幅で示す

95%信頼区間とは「同じ調査を100回やれば95回はこの区間が母平均を含む」という意味です。

## 2. 母平均の信頼区間（母分散が未知 → t分布）

$$ \bar{x} \pm t_{\,0.025,\,n-1} \times \frac{s}{\sqrt{n}} $$

ある商品の内容量を10個測ったデータで計算します。

> 📐 **数IIIメモ**：**t分布**は「正規分布に似た、すそが少し重い連続分布」。標本が小さいときの不確かさを表します。形は自由度（n−1）で決まる、とだけ分かればOK（値は `scipy` が出します）。

In [ ]:
data = np.array([198, 203, 197, 205, 199, 201, 196, 204, 200, 202])  # 標本データ
n = len(data)                   # 標本サイズ
mean = data.mean()              # 標本平均
s = data.std(ddof=1)            # 不偏標準偏差
se = s / np.sqrt(n)             # 標準誤差
t = stats.t.ppf(0.975, df=n-1)  # 自由度n-1のt値（両側95%）
low, high = mean - t*se, mean + t*se  # 信頼区間の下限・上限
print(f'標本平均: {mean:.2f}')
print(f'95%信頼区間: [{low:.2f}, {high:.2f}]')

In [ ]:
# scipyに任せると1行で95%信頼区間が出る
ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=se)
print('95%CI:', np.round(ci, 2))

> 💡 信頼度を上げる（95%→99%）と区間は**広く**なり、
サンプル数 n を増やすと区間は**狭く**なります（推定が精密に）。

In [ ]:
# 信頼度を変えると区間の幅がどう変わるか
for conf in [0.90, 0.95, 0.99]:
    lo, hi = stats.t.interval(conf, df=n-1, loc=mean, scale=se)  # 各信頼度の区間
    print(f'{int(conf*100)}%CI 幅 = {hi-lo:.2f}')   # 信頼度が高いほど幅が広い

## 3. 母比率の信頼区間

アンケートで400人中96人が「賛成」。賛成率の95%信頼区間は？

$$ \hat{p} \pm 1.96\sqrt{\frac{\hat{p}(1-\hat{p})}{n}} $$

In [ ]:
x, n = 96, 400                  # 賛成人数と回答者数
p = x / n                       # 標本比率
se = np.sqrt(p * (1 - p) / n)   # 比率の標準誤差
z = stats.norm.ppf(0.975)   # 1.96（両側95%）
print(f'賛成率の推定: {p:.3f}')
print(f'95%信頼区間: [{p - z*se:.3f}, {p + z*se:.3f}]')

> ⚠️ **よくある間違い**：95%信頼区間は「母平均がこの区間に入る確率が95%」ではありません。正しくは「同じ調査を100回やれば、95回はこの方法で作った区間が母平均を含む」という意味です。

## 4. どれだけの標本が必要？（標本サイズ設計）

「誤差をこのくらいに収めたい」と決めると、必要な標本サイズが逆算できます。母比率の95%信頼区間の半幅を `E` 以下にするには `n ≧ (1.96/E)² p(1−p)`。

In [ ]:
import numpy as np
z, E = 1.96, 0.03                 # 95%信頼・半幅3%以内
for p in [0.5, 0.3, 0.1]:
    n = (z/E)**2 * p*(1-p)
    print(f'想定 p={p}: 必要 n ≈ {np.ceil(n):.0f}')
print('→ p が不明なら p=0.5（最も n が大きい）で安全側に見積もる')

## 5. 捕獲再捕獲法 — 数えきれない総数を推定する

池の魚のように全部数えられない母集団の総数を、「印をつけて放流→再捕獲での印の割合」から推定します。

In [ ]:
rng = np.random.default_rng(0)
N_true, marked, recap = 8000, 300, 200    # 真の総数, 印をつけた数, 再捕獲数
k = rng.hypergeometric(marked, N_true-marked, recap)  # 再捕獲のうち印つき
phat = k/recap; se = np.sqrt(phat*(1-phat)/recap)
print(f'再捕獲{recap}匹中 印つき{k}匹 → 印の割合 p̂={phat:.3f}')
print(f'p の95%CI: {phat-1.96*se:.3f} 〜 {phat+1.96*se:.3f}')
print(f'総数の推定 N ≈ marked/p̂ = {marked/phat:.0f}（真の値 {N_true}）')

> 💡 標本比率の信頼区間が、そのまま総数推定の精度に効きます。再捕獲数を増やすほど区間は狭くなります。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
from scipy import stats
# Q1: 両側95%信頼区間で使う標準正規の z 値を ans に（約1.96）
ans = None   # 例: stats.norm.ppf(0.975)
_check('Q1 z(95%)', ans, stats.norm.ppf(0.975))

In [ ]:
import numpy as np
# Q2: 賛成率0.24・n=400 のときの標準誤差 √(p(1-p)/n) を ans に
ans = None
_check('Q2 標準誤差', ans, np.sqrt(0.24*0.76/400))

In [ ]:
from scipy import stats
# Q3: 信頼度を99%に上げたときの片側 z 値を ans に（95%より大きい）
ans = None   # 例: stats.norm.ppf(0.995)
_check('Q3 z(99%)', ans, stats.norm.ppf(0.995))

In [ ]:
import numpy as np
# Q4: 母比率95%CIの半幅を0.05以下にしたい。p未知(=0.5)での必要な標本サイズ n を ans に（切り上げ）
ans = None   # 例: np.ceil((1.96/0.05)**2 * 0.25)
_check('Q4 必要な標本サイズ', ans, np.ceil((1.96/0.05)**2 * 0.25))

---
## 🏆 練習問題

**問1.** あるクラス16人の小テスト平均が72点、不偏標準偏差が8点。
母平均の95%信頼区間を求めよう（t分布, 自由度15）。

**問2.** 1000人アンケートで320人が「利用したい」。利用率の95%信頼区間を求めよう。

**問3.** 問2で誤差（区間の半分）を ±2% 以内にしたい。n は何人必要か概算しよう
（ヒント：`1.96*sqrt(0.32*0.68/n) <= 0.02`）。

**問4.** 母比率の95%信頼区間の半幅を「±2%」にしたい。p=0.5として必要な標本サイズを求めよう。半幅を半分にすると必要数は何倍になる？

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


> 🔑 **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから 👉 **[01_区間推定 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/01_%E5%8C%BA%E9%96%93%E6%8E%A8%E5%AE%9A.md)**

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 点推定 | 1つの値で推定 |
| 区間推定 | 幅で推定 |
| 信頼区間 | 母数が入ると考える範囲 |
| 信頼係数 | 95%など信頼の度合い |
| 標準誤差 | 推定値の散らばり |

In [ ]:
# チートシート（区間推定）
from scipy import stats
import numpy as np
data = np.array([198,203,197,205,199,201])
se = data.std(ddof=1)/np.sqrt(len(data))
print(stats.t.interval(0.95, len(data)-1, data.mean(), se))   # 母平均の95%CI